In [ ]:
# Import dependencies
import pandas as pd
from lxml import etree
from urllib.request import urlopen
from io import BytesIO
from zipfile import ZipFile
import dns.resolver
from urllib.parse import urlparse
import tldextract

# Overheidsorganisaties
We gebruiken het [Register Internetdomeinen
Overheid](https://organisaties.overheid.nl/domeinen) om een lijst van URLs die
door de Rijksoverheid in gebruik zijn te vinden.

In [ ]:
# Load external files
rio_xml_str = urlopen("https://organisaties.overheid.nl/archive/exportRIO.xml").read()

In [ ]:
rio_xml_root = etree.fromstring(rio_xml_str)
rio_namespaces = { "p": "https://organisaties.overheid.nl/static/schema/oo/export/0.0.3" }

rio_list = []
for organisatie in rio_xml_root.iter():
    for url in organisatie.findall(".//p:domein", rio_namespaces):
        rio_list.append({ 
            "organisatie": getattr(organisatie.find(".//p:naam", rio_namespaces), "text", None),
            "url": url.find(".//p:url", rio_namespaces).text, 
        })

rio_df = pd.DataFrame(rio_list)
rio_df

In [ ]:
len(rio_df['organisatie'].unique())

In [ ]:
overheid_mx_records = []

resolver = dns.resolver.Resolver()
resolver.nameservers = ['8.8.8.8', '8.8.4.4']

for index, row in rio_df.iterrows():
    try:
        domain = tldextract.extract(row['url'])
        # Resolve MX records and get the first result
        mx_record = resolver.resolve(domain.domain + '.' + domain.suffix, 'MX')
        # Append the result (you can extract specific parts if needed)
        mx_domain = tldextract.extract(str(mx_record[0].exchange).rstrip('.'))
        overheid_mx_records.append(mx_domain.domain + '.' + mx_domain.suffix)
    except dns.resolver.NoAnswer:
        overheid_mx_records.append(None)  # No MX record found
    except Exception as e:
        overheid_mx_records.append(str(e))  # Other errors

# Add the MX records as a new column
rio_df['mx'] = overheid_mx_records

In [ ]:
c = rio_df["mx"].value_counts(dropna=False)
p = rio_df["mx"].value_counts(dropna=False, normalize=True)
pd.concat([c,p], axis=1, keys=['counts', '%'])

In [ ]:
rio_df

In [ ]:
tldextract.extract("http://mailfilter01.rekenkamer.nl")

# Bedrijven
Voor het opstellen van een lijst van bedrijven gebruiken we de [Open Data Set van
het KVK Handelsregister](https://www.kvk.nl/producten-bestellen/kvk-handelsregister-open-data-set/).

In [ ]:
kvk_bytes = urlopen("https://static.kvk.nl/download/kvk-open-data-set-handelsregister.zip").read()

In [ ]:
kvk_zip = ZipFile(BytesIO(kvk_bytes))
kvk_csv = kvk_zip.open("kvk-open-data-set-handelsregister.csv")
kvk_df = pd.read_csv(kvk_csv, on_bad_lines="skip",sep=";")
kvk_df

# Zorginstellingen
We gebruiken [Zorgkaart Nederland](https://www.zorgkaartnederland.nl) om
gegevens te verzamelen van zorgaanbieders in Nederland.

# Bedrijven v2

In [ ]:
# Load the CSV file
bedrijven = pd.read_csv('top-500-bedrijven.csv', delimiter=";")

# Iterate over the DataFrame and add the MX records
mx_records = []
for index, row in bedrijven.iterrows():
    try:
        # Resolve MX records and get the first result
        mx_record = dns.resolver.resolve(row['Domeinnaam'], 'MX')
        # Append the result (you can extract specific parts if needed)
        domain = tldextract.extract(str(mx_record[0].exchange).rstrip('.'))
        mx_records.append(domain.domain + '.' + domain.suffix)
    except dns.resolver.NoAnswer:
        mx_records.append(None)  # No MX record found
    except Exception as e:
        mx_records.append(str(e))  # Other errors

# Add the MX records as a new column
bedrijven['mx'] = mx_records

bedrijven

In [ ]:
bedrijven["mx"].value_counts()
c = bedrijven["mx"].value_counts(dropna=False)
p = bedrijven["mx"].value_counts(dropna=False, normalize=True)
pd.concat([c,p], axis=1, keys=['counts', '%'])